In [ ]:
# =============================================
# IMPORTY
# =============================================
import os
import sys
import math

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel



if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"✓ Device: {device}")
print(f"✓ PyTorch: {torch.__version__}")

# Nastav working directory na kořen projektu
_nb_dir = os.path.abspath('')
if os.path.basename(_nb_dir) in ('experiments', 'examples', 'train'):
    os.chdir(os.path.dirname(_nb_dir))
sys.path.insert(0, os.path.join(os.getcwd(), 'PFNs'))
print(f"✓ Working directory: {os.getcwd()}")
# PFNs — musí být naklonováno (viz setup.sh)
from pfns.train import train, MainConfig, OptimizerConfig, TransformerConfig, BatchShapeSamplerConfig
from pfns.model.encoders import EncoderConfig
from pfns.model.bar_distribution import BarDistributionConfig
from pfns.priors.prior import AdhocPriorConfig
from pfns.priors.fast_gp import get_batch as get_batch_for_gp

print("✓ PFNs načteny")

In [ ]:
# =============================================
# LOAD HELPER 
# =============================================

def load_pfn_model(checkpoint_path, device='cpu'):
    """
    Načte PFN model z checkpointu. Podporuje tři formáty:

    1. Nový formát (MainConfig dict, 'priors' key):
       - Uloženo přes epoch loop v PFN_TRAIN_SETUP.ipynb nebo pfn_train*.py
       - Config obsahuje celou architekturu → nlayers, emsize atd. se čtou automaticky

    2. Starý formát ('hps' key):
       - Uloženo přes train_gp_pfn()
       - Config obsahuje jen HP, architektura se musí rekonstruovat
       - nlayers se pokusí přečíst z config; pokud chybí, vyhodí chybu

    Vrací: (model, hps, epoch)
    """
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"Model nenalezen: {checkpoint_path}\n")

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config_data = checkpoint.get('config', {})

    # --- Formát 1: MainConfig dict (nový) ---
    if isinstance(config_data, dict) and 'priors' in config_data:
        saved_config = MainConfig.from_dict(config_data)
        model = saved_config.model.create_model()
        model.load_state_dict(checkpoint['model_state_dict'])
        hps = saved_config.priors[0].prior_kwargs.get('hyperparameters', {})
        epoch = checkpoint.get('epoch', '?')

    # --- Formát 2: starý formát ('hps' key) ---
    elif isinstance(config_data, dict) and 'hps' in config_data:
        hps = config_data['hps']
        epoch = config_data.get('epochs', checkpoint.get('epoch', '?'))
        criterion = checkpoint['criterion']

        nlayers = config_data.get('nlayers')
        if nlayers is None:
            # Odvoď nlayers ze state_dict — spočítej nejvyšší index vrstvy
            layer_indices = [
                int(k.split('.')[2])
                for k in checkpoint['model_state_dict']
                if k.startswith('transformer_layers.layers.')
            ]
            nlayers = max(layer_indices) + 1 if layer_indices else 6

        emsize  = config_data.get('emsize',  512)
        nhead   = config_data.get('nhead',   8)
        nhid    = config_data.get('nhid',    1024)

        cfg = MainConfig(
            priors=[AdhocPriorConfig(
                get_batch_methods=[get_batch_for_gp],
                prior_kwargs={'num_features': 1, 'hyperparameters': hps}
            )],
            optimizer=OptimizerConfig('adamw', lr=0.0003),
            model=TransformerConfig(
                criterion=BarDistributionConfig(
                    full_support=True,
                    borders=criterion.borders.tolist()
                ),
                emsize=emsize, nhead=nhead, nhid=nhid, nlayers=nlayers,
                features_per_group=1,
                attention_between_features=False,
                encoder=EncoderConfig(
                    constant_normalization_mean=0.5,
                    constant_normalization_std=math.sqrt(1/12)
                )
            ),
            batch_shape_sampler=BatchShapeSamplerConfig(
                batch_size=64, max_seq_len=50,
                min_num_features=1, max_num_features=1
            ),
            epochs=1, warmup_epochs=0,
            steps_per_epoch=1, num_workers=0,
        )
        model = cfg.model.create_model()
        model.load_state_dict(checkpoint['model_state_dict'])
        model.criterion = criterion

    else:
        raise ValueError(f"Neznámý formát checkpointu. Klíče config: {list(config_data.keys())}")

    model.to(device)
    model.eval()
    print(f"✓ Model načten: {os.path.basename(checkpoint_path)}")
    print(f"  Epocha: {epoch}, nlayers: {nlayers if 'nlayers' in dir() else '(z MainConfig)'}")
    print(f"  Parametrů: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
    return model, hps, epoch

print("✓ load_pfn_model připraven")

In [ ]:
# =============================================
# KONFIGURACE -- jedine misto k uprave
# =============================================
# POZOR: Pro Neumannuv experiment pouzij modely trenované na FIXNICH HP,
# ne na distribuci HP. Modely na distribuci HP misi dve operace:
# 1) identifikace HP z kontextu, 2) reseni lin. systemu — nelze oddelit.

MODEL_PATHS = {
    '1-layer': os.path.join('models', 'pfn_rand_hps_1layer.pth'),
    '2-layer': os.path.join('models', 'pfn_rand_hps_2layer.pth'),
    '4-layer': os.path.join('models', 'pfn_rand_hps_4layer.pth'),
    '6-layer': os.path.join('models', 'pfn_rand_hps_latest_epoch_500.pth'),
}

# HP pro GP experimenty -- musi byt konzistentni s trenovacim priorem
hps_default = {'noise': 1e-3, 'outputscale': 1.0, 'lengthscale': 0.3}

print('Nacteni modelu...')
MODELS = {}
for name, path in MODEL_PATHS.items():
    if os.path.exists(path):
        MODELS[name], _ = load_for_inference(path, device)
    else:
        print(f'  Nenalezeno: {path}')

def reset_seed():
    import random
    s = random.randint(0, 2**32 - 1)
    torch.manual_seed(s)
    np.random.seed(s % (2**31))
    if torch.backends.mps.is_available(): torch.mps.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

reset_seed()
print(f'Nacteno {len(MODELS)} modelu: {list(MODELS.keys())}')


## Experiment 2: PFN jako Neumannova řada?

### Otázka (Q2 z PFN-GP2.md)

Von Oswald et al. (2022) ukázali, že *ručně sestrojený* transformer s **lineární** attention implementuje přesně $L$ kroků gradient descent na lineárním systému. GP posterior mean vyžaduje $(K+\sigma^2 I)^{-1}y$, což GD řeší iterativně:

$$\alpha^{(0)} = 0, \qquad \alpha^{(t+1)} = \alpha^{(t)} + \eta\bigl(y - (K+\sigma^2 I)\alpha^{(t)}\bigr)$$

**Hypotéza:** Trénovaný PFN s $L$ vrstvami a softmax attention dělá něco analogického — každá vrstva přidává korekci podobnou jednomu GD kroku.

**Predikce z Neumannovy řady:** Počet kroků potřebných k dosažení chyby $\varepsilon$ je $O(\kappa \log 1/\varepsilon)$, kde podmíněnost je:

$$\kappa = \frac{\lambda_{\max}(K) + \sigma^2}{\lambda_{\min}(K) + \sigma^2}$$

| Config | $\ell$ | $\sigma^2$ | $\kappa$ | Predikovaná konvergence |
|--------|--------|------------|----------|-------------------------|
| Easy   | 0.1    | 0.5        | ~2       | 1–2 vrstvy stačí        |
| Hard   | 1.0    | 0.01       | ~100     | Potřeba mnoho vrstev    |

**Test:** Porovnáme MSE(PFN$_L$, GP) vs. MSE(GD$_L$, GP) jako funkci $L$. Pokud křivky konvergují stejnou rychlostí → PFN implementuje GD kroky. Pokud PFN konverguje rychleji → naučil se lepší optimalizátor.


In [ ]:
# =============================================
# GP UTILITY FUNKCE
# =============================================
import torch.nn.functional as F

def rbf_kernel(x1, x2, lengthscale, outputscale=1.0):
    # x1: (n,), x2: (m,) -> (n, m)
    dist_sq = (x1[:, None] - x2[None, :]) ** 2
    return outputscale * np.exp(-dist_sq / (2 * lengthscale ** 2))


def gp_exact_posterior_mean(train_x_np, train_y_np, test_x_np, lengthscale, noise_var, outputscale=1.0):
    # Presny GP posteriorni prumer: mu* = K_star (K + sigma^2 I)^{-1} y
    K = rbf_kernel(train_x_np, train_x_np, lengthscale, outputscale)
    K_noise = K + noise_var * np.eye(len(train_x_np))
    K_star = rbf_kernel(test_x_np, train_x_np, lengthscale, outputscale)
    try:
        L_chol = np.linalg.cholesky(K_noise)
        alpha = np.linalg.solve(L_chol.T, np.linalg.solve(L_chol, train_y_np))
    except np.linalg.LinAlgError:
        alpha = np.linalg.solve(K_noise, train_y_np)
    return K_star @ alpha


def condition_number(train_x_np, lengthscale, noise_var, outputscale=1.0):
    # kappa = (lambda_max(K) + sigma^2) / (lambda_min(K) + sigma^2)
    K = rbf_kernel(train_x_np, train_x_np, lengthscale, outputscale)
    eigvals = np.linalg.eigvalsh(K)
    lam_max = eigvals[-1]
    lam_min = max(eigvals[0], 0.0)  # numericky klip na 0
    kappa = (lam_max + noise_var) / (lam_min + noise_var)
    return kappa, lam_max, lam_min


def gd_iterate(train_x_np, train_y_np, test_x_np, n_steps,
               lengthscale, noise_var, outputscale=1.0, eta=None):
    # Explicitni L kroku GD: alpha^{t+1} = alpha^t + eta*(y - (K+s2I)*alpha^t)
    # Vraci list predikcí po kazdem kroku: [(pred_step0), (pred_step1), ...]
    K = rbf_kernel(train_x_np, train_x_np, lengthscale, outputscale)
    K_noise = K + noise_var * np.eye(len(train_x_np))
    K_star = rbf_kernel(test_x_np, train_x_np, lengthscale, outputscale)

    # Optimalni krok: eta = 2 / (lambda_max + lambda_min + 2*sigma^2)
    eigvals = np.linalg.eigvalsh(K_noise)
    if eta is None:
        eta = 2.0 / (eigvals[-1] + eigvals[0])

    alpha = np.zeros(len(train_y_np))
    preds = []
    for t in range(n_steps):
        residual = train_y_np - K_noise @ alpha
        alpha = alpha + eta * residual
        preds.append(K_star @ alpha)
    return preds  # list of length n_steps, each shape (n_test,)


def pfn_predict_mean(model, train_x, train_y, test_x, device):
    # Vraci mean predikci PFN z BarDistribution
    model.eval()
    with torch.no_grad():
        output = model(
            train_x.unsqueeze(0).to(device),
            train_y.unsqueeze(0).to(device),
            test_x.unsqueeze(0).to(device)
        )
        probs = torch.softmax(output, dim=-1).cpu()
        borders = model.criterion.borders.cpu()
        centers = (borders[:-1] + borders[1:]) / 2
        return (probs * centers).sum(dim=-1).squeeze().detach().cpu().numpy()

print('GP utility funkce pripraveny')


In [ ]:
# =============================================
# GENEROVANI DAT — jednou pro oba konfigurace
# =============================================

# Easy config: male lengthscale, velky sum -> mala podminenos
EASY = {'lengthscale': 0.1, 'noise': 0.5,  'outputscale': 1.0}
# Hard config: velke lengthscale, maly sum -> velka podminenos
HARD = {'lengthscale': 1.0, 'noise': 0.01, 'outputscale': 1.0}

N_CONTEXT  = 40
N_TEST     = 20
N_INSTANCES = 100

def generate_fixed_hp_datasets(n_instances, n_context, n_test, hps):
    # Generuje datasety pro fixni HP (ne z distribuce)
    datasets = []
    seq_len = n_context + n_test
    for i in range(n_instances):
        batch = get_batch_for_gp(
            batch_size=1, seq_len=seq_len, num_features=1,
            device='cpu', hyperparameters=hps
        )
        datasets.append((
            batch.x[0, :n_context],       # train_x (n_context, 1)
            batch.y[0, :n_context],        # train_y (n_context,)
            batch.x[0, n_context:],        # test_x  (n_test, 1)
        ))
    return datasets

print('Generuji Easy datasety...')
easy_datasets = generate_fixed_hp_datasets(N_INSTANCES, N_CONTEXT, N_TEST, EASY)
print('Generuji Hard datasety...')
hard_datasets = generate_fixed_hp_datasets(N_INSTANCES, N_CONTEXT, N_TEST, HARD)

# Vypocitej podminenos pro oba konfigurace (prumerem pres instance)
kappas_easy, kappas_hard = [], []
for train_x, train_y, test_x in easy_datasets[:20]:
    k, _, _ = condition_number(train_x.numpy().reshape(-1),
                               EASY['lengthscale'], EASY['noise'], EASY['outputscale'])
    kappas_easy.append(k)
for train_x, train_y, test_x in hard_datasets[:20]:
    k, _, _ = condition_number(train_x.numpy().reshape(-1),
                               HARD['lengthscale'], HARD['noise'], HARD['outputscale'])
    kappas_hard.append(k)

print(f'\nPrumerna podminenost:')
print(f'  Easy (l={EASY["lengthscale"]}, s2={EASY["noise"]}): kappa = {np.mean(kappas_easy):.1f}')
print(f'  Hard (l={HARD["lengthscale"]}, s2={HARD["noise"]}): kappa = {np.mean(kappas_hard):.1f}')
print(f'\nDatasets: {N_INSTANCES} instanci, {N_CONTEXT} context, {N_TEST} test bodu')


In [ ]:
# =============================================
# EXPERIMENT: MSE(PFN_L, GP) vs MSE(GD_L, GP)
# =============================================

def run_neumann_experiment(models_dict, datasets, hps, device,
                           max_gd_steps=8, label=''):
    """
    Pro kazdy model spocita MSE(PFN, GP).
    Pro L = 1..max_gd_steps spocita MSE(GD_L, GP).
    Vsechny modely i GD pouzivaji stejna data.

    Returns:
        pfn_mse:  dict {model_name: float}  -- prumerne MSE
        gd_mse:   dict {L: float}            -- MSE po L krocich GD
    """
    ls       = hps['lengthscale']
    noise    = hps['noise']
    oscale   = hps.get('outputscale', 1.0)
    n_instances = len(datasets)

    # --- GD iteraty ---
    gd_mse_accum = {L: [] for L in range(1, max_gd_steps + 1)}
    pfn_mse_accum = {name: [] for name in models_dict}

    for i, (train_x, train_y, test_x) in enumerate(datasets):
        if (i + 1) % 25 == 0:
            print(f'  Instance {i+1}/{n_instances}')

        train_x_np = train_x.numpy().reshape(-1)
        train_y_np = train_y.numpy().reshape(-1)
        test_x_np  = test_x.numpy().reshape(-1)

        # Ground truth: presny GP posterior
        try:
            gp_mu = gp_exact_posterior_mean(train_x_np, train_y_np, test_x_np,
                                            ls, noise, oscale)
        except Exception:
            continue

        # GD iteraty
        gd_preds = gd_iterate(train_x_np, train_y_np, test_x_np,
                              max_gd_steps, ls, noise, oscale)
        for L, pred in enumerate(gd_preds, start=1):
            mse = float(np.mean((pred - gp_mu) ** 2))
            if not np.isnan(mse) and mse < 1e4:
                gd_mse_accum[L].append(mse)

        # PFN predikce pro kazdy model
        for name, model in models_dict.items():
            try:
                pfn_mu = pfn_predict_mean(model, train_x, train_y, test_x, device)
                mse = float(np.mean((pfn_mu - gp_mu) ** 2))
                if not np.isnan(mse) and mse < 1e4:
                    pfn_mse_accum[name].append(mse)
            except Exception:
                continue

    # Prumery
    gd_mse  = {L: np.median(v) for L, v in gd_mse_accum.items() if v}
    pfn_mse = {name: np.median(v) for name, v in pfn_mse_accum.items() if v}
    pfn_err = {name: np.std(v)/np.sqrt(len(v)) for name, v in pfn_mse_accum.items() if v}

    return pfn_mse, pfn_err, gd_mse


def plot_neumann_comparison(pfn_mse, pfn_err, gd_mse, title=''):
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.tab10.colors

    # GD krivka
    ls_gd = sorted(gd_mse.keys())
    ax.plot(ls_gd, [gd_mse[l] for l in ls_gd], 's--', color='black',
            linewidth=2, markersize=8, label='GD (Neumannova rada)', zorder=5)

    # PFN body — kazdy model jako jeden bod na ose L (pocet vrstev)
    layer_map = {'1-layer': 1, '2-layer': 2, '4-layer': 4,
                 '6-layer': 6, '8-layer': 8}
    for i, (name, mse) in enumerate(pfn_mse.items()):
        L = layer_map.get(name, i + 1)
        err = pfn_err.get(name, 0)
        ax.errorbar(L, mse, yerr=err, fmt='o', color=colors[i],
                    markersize=10, capsize=5, zorder=6, label=f'PFN {name}')

    ax.set_xlabel('Pocet vrstev / GD kroku (L)', fontsize=12)
    ax.set_ylabel('MSE (vs. presny GP posterior)', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.set_yscale('log')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()

print('Funkce pripraveny')


In [ ]:
# =============================================
# SPUSTENI EXPERIMENTU
# =============================================

MAX_GD_STEPS = 8  # max pocet GD kroku (odpovidajici L vrstev)

print('=== EASY CONFIG (kappa ~ 2) ===')
pfn_easy, err_easy, gd_easy = run_neumann_experiment(
    MODELS, easy_datasets, EASY, device,
    max_gd_steps=MAX_GD_STEPS, label='Easy'
)

print('\n=== HARD CONFIG (kappa ~ 100) ===')
pfn_hard, err_hard, gd_hard = run_neumann_experiment(
    MODELS, hard_datasets, HARD, device,
    max_gd_steps=MAX_GD_STEPS, label='Hard'
)

# Ploty
plot_neumann_comparison(pfn_easy, err_easy, gd_easy,
    title=f'Easy: l={EASY["lengthscale"]}, s2={EASY["noise"]} (kappa~{np.mean(kappas_easy):.0f})')
plot_neumann_comparison(pfn_hard, err_hard, gd_hard,
    title=f'Hard: l={HARD["lengthscale"]}, s2={HARD["noise"]} (kappa~{np.mean(kappas_hard):.0f})')

# Tabulka
print('\n=== Vysledky ===')
print(f'{"Model":<12} {"MSE Easy":>12} {"MSE Hard":>12}')
print('-' * 38)
for name in MODELS:
    me = pfn_easy.get(name, float('nan'))
    mh = pfn_hard.get(name, float('nan'))
    print(f'{name:<12} {me:>12.6f} {mh:>12.6f}')
print(f'{"GD L=1":<12} {gd_easy.get(1, float("nan")):>12.6f} {gd_hard.get(1, float("nan")):>12.6f}')
print(f'{"GD L=2":<12} {gd_easy.get(2, float("nan")):>12.6f} {gd_hard.get(2, float("nan")):>12.6f}')
print(f'{"GD L=6":<12} {gd_easy.get(6, float("nan")):>12.6f} {gd_hard.get(6, float("nan")):>12.6f}')


## Co grafy ukazují

**Osa X:** počet vrstev $L$ (pro PFN) nebo počet kroků GD (pro Neumannovu řadu)

**Osa Y (log):** MSE vůči přesnému GP posterioru

**Černá přerušovaná čára (GD):** přesný výsledek $L$ kroků gradient descent — toto je to, co by PFN měl dělat podle von Oswaldovy hypotézy.

**Barevné body (PFN):** každý model s $L$ vrstvami jako jeden bod.

### Jak interpretovat:

| Pozorování | Závěr |
|---|---|
| PFN bod leží blízko GD křivky na stejném $L$ | PFN dělá přibližně GD kroky |
| PFN bod leží **pod** GD křivkou | PFN se naučil lepší optimalizátor než GD |
| PFN bod leží **nad** GD křivkou | PFN nedosahuje ani výkonu GD (kapacita nestačí) |
| Easy: rychlá konvergence, Hard: pomalá | Potvrzuje predikci $O(\kappa \log 1/\varepsilon)$ |
| Easy i Hard konvergují stejně rychle | Neumannova řada to **ne**popisuje |

### Omezení testu:

Von Oswald pracoval s **lineární** attention — u PFN s softmax se přesná korespondence nemusí držet. Navíc modely trénované na distribuci HP musí nejdřív identifikovat hyperparametry, což spotřebuje kapacitu vrstev a posune PFN body výš.
